## clode기반 rag만들기

In [ ]:
# 환경변수 불러오기
from dotenv import load_dotenv

load_dotenv()

In [ ]:
# PDF 파일 읽기
from langchain_community.document_loaders import PyPDFLoader

FILE_PATH = "가상Tech_업무가이드.pdf"

loader = PyPDFLoader(FILE_PATH)

docs = loader.load()

print(len(docs))

In [ ]:
# 페이지마다 반복되는 머리글 제거
# ("가상Tech 업무 가이드 / 사내용 · v1.0 / 문서 책임: People Operations / 페이지번호")
# 모든 청크 앞에 같은 머리글이 붙으면 임베딩이 흐려져서 검색 순위가 떨어짐
import re

HEADER = re.compile(r"^가상Tech 업무 가이드\n사내용 · v1\.0\n문서 책임: People Operations\n\d+\n")

removed = 0
for doc in docs:
    if HEADER.match(doc.page_content):
        doc.page_content = HEADER.sub("", doc.page_content)
        removed += 1

print("머리글 제거한 페이지 수:", removed)

In [ ]:
# PDF 텍스트 분할
# 머리글을 지운 docs를 잘라야 하므로 loader.load_and_split()이 아니라 split_documents(docs)를 사용
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=30,
)

split_docs = text_splitter.split_documents(docs)

print(len(split_docs))

In [7]:
# 임베딩 설정 (이 객체 하나를 저장과 검색 모두에서 재사용)
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

In [ ]:
# Pinecone 벡터 DB (클라우드) 설정
import os
import time
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore

pc = Pinecone(api_key=os.getenv("PINECONE_KEY"))   # .env의 키 이름이 PINECONE_KEY

INDEX_NAME = "rag-0928"
NAMESPACE = "rag03"   # rag_02와 청크가 달라서 네임스페이스로 구분

# 인덱스가 없을 때만 생성 (차원은 text-embedding-3-small과 같은 1536, 거리는 코사인)
if not pc.has_index(INDEX_NAME):
    pc.create_index(
        name=INDEX_NAME,
        dimension=1536,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

index = pc.Index(INDEX_NAME)
pinecone_db = PineconeVectorStore(index=index, embedding=embeddings, namespace=NAMESPACE)

def vector_count():
    return index.describe_index_stats()["namespaces"].get(NAMESPACE, {}).get("vector_count", 0)

# 비어 있을 때만 문서를 넣음
# id를 고정해 두어서, 다시 넣게 되더라도 덮어쓰기만 되고 중복이 쌓이지 않음
if vector_count() == 0:
    ids = [f"chunk-{i}" for i in range(len(split_docs))]
    pinecone_db.add_documents(split_docs, ids=ids)

    # Pinecone은 저장 직후 몇 초간 검색에 반영되지 않을 수 있어서 반영될 때까지 대기
    for _ in range(30):
        if vector_count() == len(split_docs):
            break
        time.sleep(2)
    print("새로 저장:", vector_count(), "개")
else:
    print("기존 데이터 로드:", vector_count(), "개")

In [ ]:
# 리트리버 생성 (코사인 유사도 상위 5개)
retriever = pinecone_db.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 5
    },
)

# 코사인 유사도 점수 확인 (1에 가까울수록 비슷함)
test_question = "연차를 사용하려면 최소 며칠 전에 신청해야 하나요?"
for doc, score in pinecone_db.similarity_search_with_score(test_question, k=5):
    print(f"코사인 유사도 {score:.3f} | p{int(doc.metadata['page'])} | {doc.page_content[:50]!r}")

In [10]:
# OpenAI LLM 설정
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    temperature=0,
    model="gpt-4o-mini",
)

In [11]:
# 프롬프트 (규칙은 system, 문서와 질문은 human 메시지로 분리)
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "당신은 사내 업무 가이드 문서를 근거로 답변하는 어시스턴트입니다.\n"
     "규칙:\n"
     "1. [참고 문서]에 있는 내용만 사용해서 답변하세요. 문서에 없는 내용을 추측하거나 지어내지 마세요.\n"
     "2. 질문과 문서의 표현이 달라도 같은 뜻이면 답변에 사용하세요. (예: '비용 정산' = '사용 내역 등록', '서울/제주 출장' = '국내 출장')\n"
     "3. 질문이 여러 항목을 물으면 항목별로 모두 답하세요. 계산이 필요하면 문서의 기준으로 계산해서 함께 알려주세요.\n"
     "4. 서로 다른 규정을 섞지 마세요. 예를 들어 '출발 전 신청 기한'과 '지출 후 등록(정산) 기한'은 다른 규정입니다.\n"
     "5. 문서에 관련 내용이 있으면 그 내용으로만 답하고 끝내세요. 일부 세부사항이 없더라도 '확인할 수 없다'는 문장을 덧붙이지 마세요.\n"
     "6. 관련 내용이 문서에 전혀 없을 때만 정확히 \"문서에서 확인할 수 없습니다.\"라고만 답하세요."),
    ("human", "[참고 문서]\n{context}\n\n[질문]\n{question}"),
])

chain = prompt | llm

In [12]:
# 질문
questions = [
    "연차를 사용하려면 최소 며칠 전에 신청해야 하나요?",
    "서울 출장 시 숙박비는 최대 얼마까지 지원되나요?",
    "출장을 다녀온 후 비용 정산은 언제까지 완료해야 하나요?",
    "회사 노트북을 분실했을 경우 어떤 절차로 신고해야 하나요?",
    "재택근무를 신청하려면 어떤 조건을 충족해야 하나요?",
    "직원이 업무 관련 교육을 받을 경우 회사에서 지원하는 비용 기준은 어떻게 되나요?",
    "출장 중 택시비를 회사 경비로 처리할 수 있는 경우는 언제인가요?",
    "개인 사정으로 출근 시간이 늦어질 경우 어떤 방식으로 보고해야 하나요?",
    "업무 중 시스템 장애가 발생했을 때 가장 먼저 해야 하는 조치는 무엇인가요?",
    "제주도로 2박 3일 출장을 가는 경우 숙박비 지원 한도와 출장비 정산 기한을 각각 알려주세요."
]

In [13]:
for i, question in enumerate(questions, start=1):

    # 1. 유사도 검색 (Pinecone, 코사인, 상위 5개)
    results = retriever.invoke(question)

    # 2. 검색된 문서를 하나로 합치기
    context = "\n\n".join(
        doc.page_content
        for doc in results
    )

    # 3. LLM 호출
    response = chain.invoke({"context": context, "question": question})

    # 4. 출력 (참고한 페이지도 함께, PDF 페이지 번호는 1부터)
    pages = sorted({int(doc.metadata["page"]) + 1 for doc in results})
    print(f"질문 {i}: {question}")
    print("답변:", response.content)
    print("참고 페이지:", pages)
    print()

질문 1: 연차를 사용하려면 최소 며칠 전에 신청해야 하나요?
답변: 연차를 사용하려면 최소 1일 전에 신청해야 하며, 3일 이상 사용할 경우에는 2주 전 협의가 필요합니다.
참고 페이지: [2, 6, 8, 9]

질문 2: 서울 출장 시 숙박비는 최대 얼마까지 지원되나요?
답변: 서울 출장 시 숙박비는 1박 15만원(세금 포함)까지 지원됩니다.
참고 페이지: [3, 8, 9]

질문 3: 출장을 다녀온 후 비용 정산은 언제까지 완료해야 하나요?
답변: 출장을 다녀온 후 비용 정산은 지출일로부터 5영업일 안에 ExpenseHub에 등록해야 합니다.
참고 페이지: [5, 8, 9]

질문 4: 회사 노트북을 분실했을 경우 어떤 절차로 신고해야 하나요?
답변: 문서에서 확인할 수 없습니다.
참고 페이지: [3, 5, 8]

질문 5: 재택근무를 신청하려면 어떤 조건을 충족해야 하나요?
답변: 재택근무를 신청하려면 주 2회 기준을 충족해야 하며, 매주 월요일까지 캘린더에 ‘WFH’ 표시를 해야 합니다.
참고 페이지: [2, 5, 8, 9]

질문 6: 직원이 업무 관련 교육을 받을 경우 회사에서 지원하는 비용 기준은 어떻게 되나요?
답변: 회사가 지원하는 비용 기준은 연 120만원으로, 교육·도서·컨퍼런스에 대한 지원이 포함됩니다. 신청은 LearnHub에서 사전 승인을 받아야 합니다.
참고 페이지: [2, 8]

질문 7: 출장 중 택시비를 회사 경비로 처리할 수 있는 경우는 언제인가요?
답변: 출장 중 택시비를 회사 경비로 처리할 수 있는 경우는 다음과 같습니다:

- 23:00 이후 퇴근
- 대중교통 미운행
- 고객 일정 등 업무상 필요가 있을 때

이 경우에 해당하면 택시비를 실비 지원받을 수 있습니다.
참고 페이지: [5, 8, 9]

질문 8: 개인 사정으로 출근 시간이 늦어질 경우 어떤 방식으로 보고해야 하나요?
답변: 문서에서 확인할 수 없습니다.
참고 페이지: [3, 7, 9]

질문 9: 업무 중 시스템 장애가 발생했을 때 가장 먼저 해야 하는 조